# 09 完整评分卡建模流程

从数据加载到评分卡产出，覆盖 EDA → 分箱 → 编码 → 特征筛选 → 建模 → 评估 → 评分卡。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (
    germancredit, init_setting, seed_everything,
    OptimalBinning, WOEEncoder, LogisticRegression, ScoreCard,
    IVSelector, CorrSelector, VIFSelector, CompositeFeatureSelector,
    bin_plot, ks_plot, corr_plot, plot_weights,
)
from hscredit.core.metrics import ks, auc, gini, iv_table, psi, batch_psi
from hscredit import (
    feature_summary, batch_iv_analysis, missing_analysis,
    psi_analysis, data_quality_report,
)

seed_everything(42)
init_setting()
df = germancredit()
target = 'creditability'
print(f'数据集: {df.shape}, 坏率: {df[target].mean():.2%}')

## Step 1. 数据探索 (EDA)

In [ ]:
print('=== 缺失率 ===')
print(missing_analysis(df).head(5))
print('\n=== 数据质量 ===')
print(data_quality_report(df).head(5))

In [ ]:
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
print('=== 批量IV ===')
iv_df = batch_iv_analysis(df, features=num_feats, target=target)
print(iv_df.sort_values('IV值', ascending=False))

## Step 2. 训练集/测试集划分

In [ ]:
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'训练集: {X_tr.shape}, 坏率: {y_tr.mean():.2%}')
print(f'测试集:  {X_te.shape}, 坏率: {y_te.mean():.2%}')

## Step 3. 特征筛选

In [ ]:
sel = CompositeFeatureSelector([
    ('iv',   IVSelector(threshold=0.02, max_n_bins=5)),
    ('corr', CorrSelector(threshold=0.85)),
    ('vif',  VIFSelector(threshold=10)),
])
sel.fit(X_tr, y_tr)
X_tr_sel = sel.transform(X_tr)
X_te_sel  = sel.transform(X_te)
sel_feats = X_tr_sel.columns.tolist()
print(f'原始特征数: {X_tr.shape[1]}, 筛选后: {len(sel_feats)}')
print('保留特征:', sel_feats)

## Step 4. 最优分箱

In [ ]:
binner = OptimalBinning(method='mdlp', max_n_bins=6)
binner.fit(X_tr_sel, y_tr)

print('各特征分箱IV:')
for feat, bt in binner.bin_tables_.items():
    if '分档IV值' in bt.columns:
        print(f'  {feat}: IV={bt["分档IV值"].sum():.4f}, 箱数={len(bt)}')

In [ ]:
# 可视化第一个特征的分箱
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig = bin_plot(binner.bin_tables_[sel_feats[0]], desc=sel_feats[0])
plt.show()

## Step 5. WOE 编码

In [ ]:
woe = WOEEncoder(max_n_bins=6)
X_woe_tr = woe.fit_transform(X_tr_sel, y_tr)
X_woe_te  = woe.transform(X_te_sel)
print('WOE编码后:')
X_woe_tr.head(3)

## Step 6. 逻辑回归训练

In [ ]:
lr = LogisticRegression(C=0.5, max_iter=1000, calculate_stats=True)
lr.fit(X_woe_tr, y_tr)
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
y_prob_te  = lr.predict_proba(X_woe_te)[:,1]
print('--- 训练集 ---')
print(f'KS: {ks(y_tr, y_prob_tr):.4f}, AUC: {auc(y_tr, y_prob_tr):.4f}')
print('--- 测试集 ---')
print(f'KS: {ks(y_te, y_prob_te):.4f}, AUC: {auc(y_te, y_prob_te):.4f}')

In [ ]:
print('模型系数摘要:')
lr.summary()

In [ ]:
fig = plot_weights(lr); plt.show()

## Step 7. KS & ROC 评估

In [ ]:
fig = ks_plot(y_prob_te, y_te, title='测试集 KS/ROC'); plt.show()

## Step 8. 生成评分卡

In [ ]:
sc = ScoreCard(pdo=20, score=600, odds=1/19,
    lr_kwargs={'C':0.5,'max_iter':1000,'calculate_stats':True})
sc.fit(X_woe_tr, y_tr, woe_encoder=woe)
scores_te = sc.predict(X_woe_te)
print('测试集评分分布:')
print(pd.Series(scores_te).describe().round(1))

In [ ]:
print('评分卡表:')
sc.scorecard_table_.head(12)

## Step 9. 评分 PSI 检验

In [ ]:
scores_tr = sc.predict(X_woe_tr)
score_psi = psi(scores_tr, scores_te)
print(f'评分PSI: {score_psi:.4f}')

## Step 10. 特征PSI监控

In [ ]:
psi_monitor = batch_psi(X_tr_sel, X_te_sel, features=sel_feats[:4])
print(psi_monitor)